<a href="https://colab.research.google.com/github/sankethashegde/AML-lab/blob/main/Copy_of_lab_09_rnn_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-09-rnn-lstm.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 9 — Sequence models: RNN, LSTM and GRU on Shakespeare

**MSIS · AME 5053 · Week 9 · 3 hours**

Lab 5 built a language model by counting. Its bigram model predicted the next word from the one
word before it, and nothing earlier: after *the*, it gave the same guesses whether the sentence
was about a queen or a rabbit-hole. That is the Markov assumption, and lab 5 made it because
counting cannot do better without running out of data.

This lab keeps the task and removes the assumption. We train a recurrent network to read
Shakespeare one character at a time and predict the next character. The bigram model, applied to
characters, is the baseline: it sees one character back. The RNN of session 23 carries a hidden
state forward, so its guess can depend on everything it has read. We measure what that memory is
worth, watch it fade in the way session 24 predicted, and then swap in the LSTM of session 25 and
the GRU of session 26. The last two parts repeat the experiment with whole words, which brings
back lab 8's embeddings.

**By the end of this lab you will be able to:**

1. Turn a text into training pairs for a language model: the target is the input shifted by one
2. Build a character-level language model around `nn.RNN`, and name the matrices PyTorch keeps
3. Train it with backpropagation through time and gradient clipping, and read the loss
4. Measure the vanishing gradient directly, and see what the LSTM and GRU change about it
5. Sample text from a trained model, and say what temperature does
6. Move from characters to words with an embedding layer, and inspect what the embedding learned

The code in this lab is the code in the course's RNN repository, <https://github.com/shripada/rnn>, which has
`demo/char_rnn.py` and `demo/word_rnn.py` as command-line scripts you can run on your own laptop.
The lecture decks that go with it are [Part 1 · Networks that remember](https://shripada.github.io/rnn/slides/rnn-intro.html)
and [Part 2 · Memory with a door](https://shripada.github.io/rnn/slides/lstm-gru.html). Both have interactive versions of the
experiments in this lab.

Run every cell, in order.

---

## Part 0 — Setup

PyTorch is already installed on Colab, and nothing else is needed. The text is downloaded from
the `karpathy/char-rnn` repository on GitHub.

In [ ]:
import copy
import math
import re
import time
import urllib.request
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F

print("torch", torch.__version__, "· threads", torch.get_num_threads())

torch 2.14.1+cpu · threads 1


### Where this will run

The lab works with or without a GPU. If Colab has given you one (Runtime → Change runtime type →
T4 GPU) the models train there; if not, everything runs on the CPU and the only difference is how
long you wait. On a laptop CPU the three character models take about 25 seconds, 80 seconds and
65 seconds, and the word model about 80.

The numbers quoted in this lab come from one run on a laptop CPU. On a GPU, or on a different
CPU, your numbers will land within about a point of them, because the arithmetic is not
bit-for-bit identical. What the lab measures — which model beats which, and by how much — does
not change.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training on:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

training on: cpu


> **Save your own copy now:** File → Save a copy in Drive.

---

## Part 1 — The text, as numbers

Our corpus is `tinyshakespeare`: about 1.1 million characters taken from Shakespeare's plays,
speaker names in capitals, one line of verse per line.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
with urllib.request.urlopen(DATA_URL) as f:
    text = f.read().decode("utf-8")

print(f"{len(text):,} characters\n")
print(text[:300])

1,115,394 characters

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


### Train, validation and test

We split the text by position, not at random: the first 80% to train on, the next 10% to check
progress during training, and the last 10% held back as the final exam. A random split would
put lines from the same speech on both sides, and the test would be easier than any text the
model meets afterwards. Splitting by position means the test text is whole scenes the model has
never read.

In [ ]:
def split_text(text: str):
    """TRAIN / VAL / TEST, 80 / 10 / 10, in order."""
    a, b = int(0.8 * len(text)), int(0.9 * len(text))
    return text[:a], text[a:b], text[b:]

train_text, val_text, test_text = split_text(text)
print(f"TRAIN {len(train_text):,} · VAL {len(val_text):,} · TEST {len(test_text):,} characters")

TRAIN 892,315 · VAL 111,539 · TEST 111,540 characters


### The vocabulary is the alphabet

A network works with numbers, so each distinct character gets an integer id. The text has 65 of
them: letters in both cases, digits, punctuation, space and newline. With characters as the unit
there is no unknown-word problem — every character the model will ever read is one of these 65.

In [ ]:
class Vocab:
    """Every distinct character in the text, with an integer id for each."""

    def __init__(self, chars):
        self.chars = list(chars)                                # id -> char
        self.stoi = {c: i for i, c in enumerate(self.chars)}    # char -> id

    @classmethod
    def from_text(cls, text: str):
        return cls(sorted(set(text)))

    def encode(self, s: str) -> torch.Tensor:
        unknown = sorted(set(c for c in s if c not in self.stoi))
        if unknown:
            raise ValueError(f"These characters never appear in the text, so the model can't read them: {unknown}")
        return torch.tensor([self.stoi[c] for c in s], dtype=torch.long)

    def decode(self, ids) -> str:
        return "".join(self.chars[i] for i in ids)

    def __len__(self):
        return len(self.chars)

vocab = Vocab.from_text(text)
train, val = vocab.encode(train_text), vocab.encode(val_text)
print(len(vocab), "characters:", repr("".join(vocab.chars)))
print("'ROMEO' ->", vocab.encode("ROMEO").tolist())

65 characters: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
'ROMEO' -> [30, 27, 25, 17, 27]


### The target is the input, shifted by one

A language model needs no labels beyond the text itself. We cut a chunk out of the text and use
it as the input $x$; the target $y$ is the same chunk moved one character to the right. At every
position $t$ the job is the same: having read $x_1 \dots x_t$, predict $x_{t+1}$.

    x:  F  i  r  s  t     C  i  t
    y:  i  r  s  t     C  i  t  i

Every position has its own target. In session 23's terms this is sequence labelling, where the
label at each step is simply the next character, so one 100-character chunk gives 100 training
examples at once.

In [ ]:
def get_batch(data: torch.Tensor, batch_size: int, seq_len: int):
    """`batch_size` random chunks of `seq_len` ids, and the same chunks shifted by one.
    Returns two integer tensors of shape (batch_size, seq_len)."""
    starts = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([data[s : s + seq_len] for s in starts])
    y = torch.stack([data[s + 1 : s + seq_len + 1] for s in starts])
    return x, y

x, y = get_batch(train, 1, 40)
print("input  x:", repr(vocab.decode(x[0].tolist())))
print("target y:", repr(vocab.decode(y[0].tolist())))

input  x: 'f\nThat swells with silence in the tortur'
target y: '\nThat swells with silence in the torture'


---

## Part 2 — Lab 5's bigram model, on characters

Before training anything we need to know what a model with no memory achieves. Lab 5's bigram
model is exactly that model: count, in the training text, which character follows which, and
guess the most frequent follower of the last character. It looks one character back and no
further.

In [ ]:
class LastCharOnly:
    """A character bigram model: guess the next character from the last one alone,
    using counts from TRAIN. The best you can do with no memory."""

    def __init__(self, text: str):
        self.counts = defaultdict(Counter)
        for a, b in zip(text, text[1:]):
            self.counts[a][b] += 1

    def top_guesses(self, prompt: str, top: int = 3):
        c = self.counts[prompt[-1]]
        total = sum(c.values()) or 1
        return [(ch, n / total) for ch, n in c.most_common(top)]

    def best_guess(self, ch: str) -> str:
        c = self.counts[ch]
        return c.most_common(1)[0][0] if c else " "

baseline = LastCharOnly(train_text)


def show(ch: str) -> str:
    """Make whitespace visible when printing a single character."""
    return {" ": "␣", "\n": "↵"}.get(ch, ch)


def fmt_guesses(guesses):
    return ", ".join(f"{show(c)} {p:.0%}" for c, p in guesses)


for prompt in ["ROME", "KING RICHAR"]:
    print(f"{prompt!r:15} -> {fmt_guesses(baseline.top_guesses(prompt, 5))}")

'ROME'          -> N 21%, S 12%, R 12%, T 9%, : 8%
'KING RICHAR'   -> I 17%, D 14%, E 7%, o 7%, : 7%


After `ROME` the bigram model sees only the capital `E`, and after a capital `E` the commonest
characters in this text are `N` (21%), `S`, `R`, `T` and a colon. `O` is not among them, because
nothing about `E` alone says that the word is *Romeo*. This is lab 5's failure in miniature: the information that decides
the answer is further back than the model can see.

How good is this model on the test text? We grade every position of the TEST text by whether the
single best guess is exactly the next character.

In [ ]:
test_data = vocab.encode(test_text)
L = 200                                    # grade the test text in back-to-back 200-char chunks
n_chunks = (len(test_data) - 1) // L
test_x = test_data[: n_chunks * L].view(n_chunks, L)
test_y = test_data[1 : n_chunks * L + 1].view(n_chunks, L)

base_pred = torch.tensor([[vocab.stoi[baseline.best_guess(vocab.chars[c])] for c in row]
                          for row in test_x.tolist()])
base_acc = (base_pred == test_y).float().mean().item()
print(f"graded on {n_chunks * L:,} test characters")
print(f"  random guessing (1 in {len(vocab)}):  {1 / len(vocab):6.1%}")
print(f"  bigram, last character only:  {base_acc:6.1%}")

graded on 111,400 test characters
  random guessing (1 in 65):    1.5%
  bigram, last character only:   26.5%


The bigram model gets 26.5% right, against 1.5% for guessing at random. That is the number to beat.
Anything a recurrent network scores above it comes from remembering characters further back.

---

## Part 3 — The RNN

Session 23 gave the simple recurrent network in SLP3's notation (§13.1):

$$h_t = g(U h_{t-1} + W x_t) \qquad y_t = f(V h_t)$$

Here $x_t$ is the character at step $t$, written as a one-hot vector of length 65; $h_t$ is the
hidden state, of size 256 in this lab; $g$ is $\tanh$; and $y_t$ is a score for each of the 65
possible next characters, which softmax turns into probabilities. The same $U$, $W$ and $V$ are
used at every step, so the model can read text of any length with a fixed number of weights.

`nn.RNN` computes $h_1, \dots, h_T$ for a whole chunk in one call. PyTorch's names for the
matrices are not SLP3's:

| SLP3 | PyTorch | shape |
|---|---|---|
| $W$ (input → hidden) | `weight_ih_l0` | 256 × 65 |
| $U$ (hidden → hidden, the recurrence) | `weight_hh_l0` | 256 × 256 |
| $V$ (hidden → output) | `head.weight`, our `nn.Linear` | 65 × 256 |

PyTorch also keeps two bias vectors, `bias_ih_l0` and `bias_hh_l0`. They are added together, so
they act as one bias.

The class below builds all three cells this lab uses. Only the line that creates `self.rnn`
changes between them, and Parts 4 and 5 return to the other two.

In [ ]:
CELLS = {"rnn": "vanilla RNN", "lstm": "LSTM", "gru": "GRU"}


class CharRNN(nn.Module):
    """x_t one-hot (65) -> h_t (hidden) -> scores for the next character (65)."""

    def __init__(self, vocab_size: int, hidden: int, cell: str = "rnn"):
        super().__init__()
        self.vocab_size = vocab_size
        self.cell = cell
        if cell == "rnn":
            # Session 23: h_t = tanh(W x_t + U h_{t-1} + b).
            self.rnn = nn.RNN(input_size=vocab_size, hidden_size=hidden,
                              nonlinearity="tanh", batch_first=True)
        elif cell == "lstm":
            # Session 25: four weight sets (forget, input, candidate, output) stacked into one
            # matrix, so weight_ih_l0 is (4n x d). The state is a pair (h_t, c_t).
            self.rnn = nn.LSTM(input_size=vocab_size, hidden_size=hidden, batch_first=True)
        elif cell == "gru":
            # Session 26: three weight sets (reset, update, candidate). The state is just h_t.
            self.rnn = nn.GRU(input_size=vocab_size, hidden_size=hidden, batch_first=True)
        else:
            raise ValueError(f"unknown cell {cell!r}")
        self.head = nn.Linear(hidden, vocab_size)     # V: hidden state -> 65 scores

    def forward(self, x, h=None):
        """x: (batch, time) character ids. h: the state to start from; None means h_0 = 0.
        Returns scores for the next character at every step, (batch, time, 65), and the
        final state, so that reading can continue where it stopped."""
        x_onehot = F.one_hot(x, self.vocab_size).float()   # (batch, time, 65)
        all_h, h = self.rnn(x_onehot, h)                   # all_h holds h_1 .. h_T
        return self.head(all_h), h


m = CharRNN(len(vocab), 256, "rnn")
for name, p in m.named_parameters():
    print(f"{name:14} {tuple(p.shape)}")
print(f"{sum(p.numel() for p in m.parameters()):,} weights")

rnn.weight_ih_l0 (256, 65)
rnn.weight_hh_l0 (256, 256)
rnn.bias_ih_l0 (256,)
rnn.bias_hh_l0 (256,)
head.weight    (65, 256)
head.bias      (65,)
99,393 weights


### What one training round does

Each round of training has five steps, and the loop in `train_char` below is commented with them:

1. **Guess.** Take 64 random 100-character chunks of TRAIN and let the RNN read them, guessing
   the next character at every position.
2. **Check.** Compare every guess with the real next character. The loss is cross-entropy: the
   negative log of the probability the model gave to the right answer. Guessing uniformly among
   65 characters gives $-\log(1/65) = 4.17$, so that is where training starts.
3. **Assign the blame.** `loss.backward()` is backpropagation through time, session 24: autograd
   walks back through the chain of 100 unrolled steps and works out, for every weight, which way
   to move it. Reading the whole play as one sequence would be far too slow, so we use
   **truncated backpropagation through time**: the gradient flows back to the start of the
   100-character chunk and no further.
4. **Clip.** Session 24 showed that the product which makes gradients vanish can also make them
   explode. **Gradient clipping** is the standard guard: if the gradient's norm exceeds a
   threshold, here 1, it is scaled down to that threshold before the weights move.
5. **Adjust.** The optimiser, Adam, moves every weight a small step in the direction that lowers
   the loss.

Every 200 rounds we also measure the loss on a VAL batch, which the model never trains on. If
train loss keeps falling while val loss does not, the model is memorising rather than learning.
At rounds 1000 and 2000 it writes 150 characters, so you can watch it learn the shape of a play.

The gradient probe that runs before and after training is the subject of Part 4; for now, notice
that it prints two tables.

In [ ]:
@torch.no_grad()
def generate_text(model, vocab, prompt: str, n: int, temperature: float = 0.8) -> str:
    """Read the prompt, then sample a character, feed it back in as the next input, and repeat."""
    was_training = model.training
    model.eval()
    logits, h = model(vocab.encode(prompt).unsqueeze(0).to(DEVICE))     # read the prompt
    out = list(prompt)
    for _ in range(n):
        probs = torch.softmax(logits[0, -1] / temperature, dim=-1)
        nxt = torch.multinomial(probs, 1).item()                         # sample one character
        out.append(vocab.chars[nxt])
        logits, h = model(torch.tensor([[nxt]], device=DEVICE), h)       # carry h forward
    model.train(was_training)
    return "".join(out)


def gradient_probe(model, data: torch.Tensor, seq_len: int = 100, batch: int = 32):
    """Put a loss on the LAST prediction of a 100-step chunk only, backpropagate, and
    return |dL/dh_t| for every step t: how much of the learning signal reaches each step."""
    x, y = get_batch(data, batch, seq_len)
    x, y = x.to(DEVICE), y.to(DEVICE)
    x_onehot = F.one_hot(x, model.vocab_size).float()
    state, hs = None, []
    for t in range(seq_len):
        _, state = model.rnn(x_onehot[:, t : t + 1], state)   # one step: h_{t-1} -> h_t
        h = state[0] if model.cell == "lstm" else state      # the LSTM's state is (h, c)
        h.retain_grad()                                       # keep dL/dh_t so we can read it
        hs.append(h)
    loss = F.cross_entropy(model.head(hs[-1][0]), y[:, -1])
    model.zero_grad()
    loss.backward()
    norms = [s.grad.norm().item() / batch for s in hs]
    model.zero_grad()                                         # leave no trace on training
    return norms


def print_probe(norms, title):
    print(f"\n{title}")
    print("  steps back   |dL/dh|    (bar: log scale, 1 '#' ≈ ×2.5)")
    T, top = len(norms), max(norms)
    for k in [0, 1, 2, 5, 10, 20, 40, 60, 99]:
        if k < T:
            v = norms[T - 1 - k]
            bar = "#" * max(0, int(25 + math.log10(max(v / top, 1e-30)) * 2.5))
            print(f"  {k:>10}   {v:9.2e}  {bar}")


def train_char(cell="rnn", steps=2000, hidden=256, seq_len=100, batch=64, lr=3e-3, clip=1.0, seed=0):
    torch.manual_seed(seed)
    x, y = get_batch(train, 1, 40)            # drawn so the random numbers match char_rnn.py
    model = CharRNN(len(vocab), hidden, cell).to(DEVICE)
    print(f"{CELLS[cell]}, hidden size {hidden}, {sum(p.numel() for p in model.parameters()):,} weights")
    probe_before = gradient_probe(model, train)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    print(f"\n{'round':>6} {'train loss':>11} {'val loss':>9} {'grad norm':>10} {'clipped':>8}")
    t0, clipped = time.time(), 0
    for step in range(1, steps + 1):
        x, y = get_batch(train, batch, seq_len)                       # (1) guess
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, _ = model(x)
        loss = loss_fn(logits.reshape(-1, len(vocab)), y.reshape(-1))  # (2) check
        opt.zero_grad()
        loss.backward()                                               # (3) blame: BPTT
        norm = nn.utils.clip_grad_norm_(model.parameters(), clip).item()   # (4) clip
        clipped += norm > clip
        opt.step()                                                    # (5) adjust

        if step % 200 == 0 or step == 1:
            with torch.no_grad():
                vx, vy = get_batch(val, batch, seq_len)
                vx, vy = vx.to(DEVICE), vy.to(DEVICE)
                vloss = loss_fn(model(vx)[0].reshape(-1, len(vocab)), vy.reshape(-1)).item()
            print(f"{step:>6} {loss.item():>11.3f} {vloss:>9.3f} {norm:>10.2f} {clipped:>8}")
            clipped = 0
        if step % 1000 == 0 or step == steps:
            print(f"  ── what it writes after {step} rounds (prompt 'ROMEO:') ──")
            print("  " + generate_text(model, vocab, "ROMEO:\n", 150).replace("\n", "\n  "))
            print("  " + "─" * 48)

    print(f"\nTrained in {time.time() - t0:.0f}s.")
    model.probe = (probe_before, gradient_probe(model, train))
    print_probe(model.probe[0], f"Gradient probe BEFORE training ({CELLS[cell]}):")
    print_probe(model.probe[1], f"Gradient probe AFTER training ({CELLS[cell]}):")
    model.eval()
    return model

In [ ]:
char_models = {}
char_models["rnn"] = train_char("rnn")

vanilla RNN, hidden size 256, 99,393 weights

 round  train loss  val loss  grad norm  clipped
     1       4.171     4.044       0.37        0
   200       2.140     2.198       0.71       37
   400       1.872     1.960       0.49       18
   600       1.801     1.867       0.81        2
   800       1.641     1.741       0.73        4
  1000       1.620     1.782       0.62        1
  ── what it writes after 1000 rounds (prompt 'ROMEO:') ──
  ROMEO:
  So your compertall with his shall be not bentence that mores it me!
  
  COMINIUS:
  My lorethy with the coar mine own so houme, themel should no host but a
  ────────────────────────────────────────────────
  1200       1.560     1.670       0.63        1
  1400       1.536     1.725       0.50        2
  1600       1.545     1.696       0.55        3
  1800       1.522     1.705       0.52        2
  2000       1.501     1.663       0.58        3
  ── what it writes after 2000 rounds (prompt 'ROMEO:') ──
  ROMEO:
  And, and I'll powe

### Reading the training log

The loss starts at 4.17, uniform guessing over 65 characters, and falls to about 1.5. A useful way
to read a cross-entropy loss is $e^{\text{loss}}$: the number of characters the model is still,
in effect, choosing between. $e^{1.5} \approx 4.5$, against 65 at the start.

Val loss stays within about 0.2 of train loss, and is still falling at round 2000, so the model
has not started memorising. The `clipped` column counts how many rounds in each block of 200 had
their gradient scaled down: 42 in the first block of the reference run, 16 in the second, then a
handful.

The samples are the more vivid record. After 1000 rounds the model already writes speaker names
in capitals followed by a colon and a newline, and something with the rhythm of verse. It has
learned spelling and layout. It has not learned meaning, and a model of this size trained for
half a minute will not.

### Grading it on the test text

`test_char` grades every position of TEST, exactly as we graded the bigram model, and prints a
few test lines with the model's guess under each character. A `·` means the guess was right.

In [ ]:
@torch.no_grad()
def test_char(model, seed=0):
    logits, _ = model(test_x.to(DEVICE))
    logits = logits.cpu()
    pred = logits.argmax(-1)
    loss = F.cross_entropy(logits.reshape(-1, len(vocab)), test_y.reshape(-1)).item()
    acc = (pred == test_y).float().mean().item()
    print("How often is the top guess for the next character exactly right?")
    print(f"  random guessing (1 in {len(vocab)}):  {1 / len(vocab):6.1%}")
    print(f"  bigram, last character only:  {base_acc:6.1%}   (no memory)")
    print(f"  {CELLS[model.cell] + ':':<29}{acc:6.1%}   (loss {loss:.2f})")

    g = torch.Generator().manual_seed(seed)
    for _ in range(4):
        r = torch.randint(0, n_chunks, (1,), generator=g).item()
        start = torch.randint(0, L - 60, (1,), generator=g).item()
        actual = vocab.decode(test_y[r, start : start + 60].tolist())
        guess = vocab.decode(pred[r, start : start + 60].tolist())
        marks = "".join("·" if a == b else show(b) for a, b in zip(actual, guess))
        print(f"\n  actual : {''.join(show(c) for c in actual)}")
        print(f"  guessed: {marks}")
    return acc

char_acc = {"rnn": test_char(char_models["rnn"])}

How often is the top guess for the next character exactly right?
  random guessing (1 in 65):    1.5%
  bigram, last character only:   26.5%   (no memory)
  vanilla RNN:                  47.2%   (loss 1.85)

  actual : ETRUCHIO:↵Did␣ever␣Dian␣so␣become␣a␣grove↵As␣Kate␣this␣chamb
  guessed: OREGSUESR·Wo··tl·n·wodtct··mrsi···onfoaa·␣Tn·minh·I·e··po·n·

  actual : gainst␣the␣bias.↵But,␣soft!␣company␣is␣coming␣here.↵Good␣mor
  guessed: n··········wedt␣·↵··␣·ai␣f␣·I··e··s·tn·a··e···t··␣␣·↵·····a·

  actual : issued.↵↵MIRANDA:↵O␣the␣heavens!↵What␣foul␣play␣had␣we,␣that
  guessed: tn␣··␣␣··KESC·C:··T,h·o·p·····␣␣·↵·e··s·r··tr·c·tit·t·l···e·

  actual : us␣aboard␣a␣bark,↵Bore␣us␣some␣leagues␣to␣sea;␣where␣they␣pr
  guessed: i··tnsu·␣etng··d␣␣Autd·t··a·␣··a··v··␣··h·m·e␣·I·an··i··␣·s·


The RNN gets 47.1% of test characters right against the bigram model's 26.5%. Both models were
trained on the same text and graded on the same text; the only thing the RNN has that the bigram
model lacks is $h_{t-1}$. So the twenty-point gap is a measurement of what the hidden state is
worth on this task.

The guessed lines show where the remaining errors are. The model is reliable inside a word,
where the next letter is nearly determined, and wrong at the start of one, where many words are
possible.

**Try it.** Edit the prompt in the cell below and run it. A newline is written `\n`, so
`ROMEO:\n` asks what comes at the start of Romeo's speech. Compare the RNN's guesses with the
bigram model's: where they disagree, the RNN is using something further back than the last
character.

In [ ]:
PROMPT = "KING RICHAR"  # @param {type:"string"}
CELL = "rnn"  # @param ["rnn", "lstm", "gru"]


@torch.no_grad()
def top_guesses(model, vocab, prompt: str, top: int = 3):
    """Read the whole prompt and return the top guesses for the next character.
    Only the last step's output is used."""
    logits, _ = model(vocab.encode(prompt).unsqueeze(0).to(DEVICE))
    probs = torch.softmax(logits[0, -1], dim=-1)
    p, ids = probs.topk(top)
    return [(vocab.chars[i], pi.item()) for i, pi in zip(ids.tolist(), p)]


prompt = PROMPT.replace("\\n", "\n")
model = char_models.get(CELL)
if model is None:
    print(f"No {CELL} model yet: it is trained further down. Using the RNN.")
    model = char_models["rnn"]
print(f"Text so far: {prompt!r}\n")
print(f"  {CELLS[model.cell] + ':':<36}{fmt_guesses(top_guesses(model, vocab, prompt, 5))}")
print(f"  bigram, last character only:        {fmt_guesses(baseline.top_guesses(prompt, 5))}")

Text so far: 'KING RICHAR'

  vanilla RNN:                        D 99%, E 0%, W 0%, C 0%, G 0%
  bigram, last character only:        I 17%, D 14%, E 7%, o 7%, : 7%


---

## Part 4 — Watching the gradient vanish

Session 24 showed why a plain RNN struggles to learn from far back. The gradient that reaches
$h_1$ from a loss at step $n$ passes through the recurrence $n-1$ times, so in the linear case
it is multiplied by $U^{\,n-1}$; when that factor is below 1 it shrinks geometrically. The RNN
can only learn to use a character if some gradient from the place where that character matters
gets back to it.

`gradient_probe` measures this directly. It runs the model over a 100-character chunk one step at
a time, puts a loss on the final prediction only, backpropagates, and records the size of
$\partial L / \partial h_t$ at every step. The two tables `train_char` printed are that
measurement, before and after training. Here are the trained model's numbers again, at a few
distances.

In [ ]:
def probe_row(model, ks=(1, 10, 40, 99)):
    norms = model.probe[1]
    T = len(norms)
    return [norms[T - 1 - k] for k in ks]

print("RNN, after training. |dL/dh| at k steps back:")
for k, v in zip((1, 10, 40, 99), probe_row(char_models["rnn"])):
    print(f"  {k:>3} steps back: {v:.1e}")

RNN, after training. |dL/dh| at k steps back:
    1 steps back: 9.7e-03
   10 steps back: 7.7e-04
   40 steps back: 8.3e-07
   99 steps back: 3.4e-12


Forty steps back, the signal is about 1e−5 of a loss that was itself averaged over the batch;
ninety-nine steps back it is about 1e−9. Each `#` in the probe's bar chart is a factor of about
2.5, and the bars shorten steadily with distance: the geometric decay session 24 derived, now
measured on a real model. In practice the RNN learns from the last ten or twenty characters and
very little before that, which is enough for spelling and not enough for who is speaking.

---

## Part 5 — The LSTM and the GRU

Session 25's LSTM replaces the multiplicative path through $U$ with an additive one. Its
equations (13.21), (13.24) and (13.25) put the new context together as

$$c_t = c_{t-1} \odot f_t + g_t \odot i_t$$

so a gradient flowing back along $c$ is multiplied by the forget gate $f_t$, which the network can
learn to hold near 1, rather than by $U$ and a $\tanh$ derivative at every step. Session 26's GRU
does the same job with two gates and no separate cell state.

In PyTorch the swap is one word: `nn.LSTM` or `nn.GRU` in place of `nn.RNN`, with the same
arguments. The only other change is that the LSTM's state is a pair $(h_t, c_t)$, which
`CharRNN.forward` and `gradient_probe` already handle. The LSTM has four weight sets where the
RNN had one, so it has three and a half times the weights and takes about three times as long to
train.

Session 26's caution applies to the GRU we train here: `nn.GRU` is not Cho's equation (8) as
published, because PyTorch applies the reset gate after the weight matrix rather than before. What
this lab measures is PyTorch's variant.

In [ ]:
char_models["lstm"] = train_char("lstm")
char_acc["lstm"] = test_char(char_models["lstm"])

LSTM, hidden size 256, 347,457 weights

 round  train loss  val loss  grad norm  clipped
     1       4.157     4.113       0.23        0


KeyboardInterrupt: 

In [ ]:
char_models["gru"] = train_char("gru")
char_acc["gru"] = test_char(char_models["gru"])

### The three cells side by side

In [ ]:
print(f"{'':30}{'vanilla RNN':>13}{'LSTM':>10}{'GRU':>10}")
print(f"{'weights':30}" + "".join(f"{sum(p.numel() for p in char_models[c].parameters()):>{w},}"
                                   for c, w in (("rnn", 13), ("lstm", 10), ("gru", 10))))
print(f"{'test: next char right':30}" + "".join(f"{char_acc[c]:>{w}.1%}"
                                                 for c, w in (("rnn", 13), ("lstm", 10), ("gru", 10))))
for k, i in ((40, 2), (99, 3)):
    print(f"{f'probe: gradient {k} steps back':30}" +
          "".join(f"{probe_row(char_models[c])[i]:>{w}.1e}" for c, w in (("rnn", 13), ("lstm", 10), ("gru", 10))))
print(f"\nbigram, last character only: {base_acc:.1%}")

From the reference run:

| | vanilla RNN | LSTM | GRU |
|---|---|---|---|
| weights | 99,393 | 347,457 | 264,769 |
| training time (laptop CPU) | 24 s | 80 s | 64 s |
| test: next char right | 47.1% | 50.0% | 50.3% |
| probe: gradient 40 steps back | 1.1e−5 | 2.8e−4 | 1.4e−3 |
| probe: gradient 99 steps back | 2.3e−9 | 2.7e−6 | 6.1e−7 |

### Reading the comparison

The gated cells keep far more gradient alive at a distance: forty steps back, 25 times more for
the LSTM and over 100 times more for the GRU. That is session 25's argument, and here it is a
measurement rather than a derivation. It buys about three points of accuracy, which is real but
modest. Most of the next character in English is decided by the last few, and the plain RNN
already handles those.

Which of the LSTM and GRU is better, this table cannot say. The gap between them is 0.3 points
from one run with one seed. Session 26 cited Chung et al. (2014), who compared the two across
several tasks and found them about equal; one run here is no evidence against that. The GRU
trains a little faster because it has three weight sets to the LSTM's four.

**Try it.** Go back to the prompt cell at the end of Part 3, set `CELL` to `lstm` or `gru`, and
run it again.

---

## Part 6 — Letting the model write

To generate text, `generate_text` reads a prompt, turns the last output into probabilities,
samples a character from them, feeds that character back in as the next input, and repeats. This
is how lab 5 generated text from its bigram counts, with the bigram table replaced by the
network.

We sample rather than always taking the most likely character, because the most likely choice
tends to loop (*the the the*). The **temperature** $\tau$ divides the scores before the softmax,
$\mathrm{softmax}(y_t / \tau)$. Below 1 it sharpens the distribution and the text becomes safer
and more repetitive; above 1 it flattens it and the text becomes stranger. At $\tau = 1$ we sample
from the model's own probabilities.

**Try it.** Change the prompt, the cell, the length and the temperature, and run the cell again.
Try a temperature of 0.3 and then 1.5 with everything else fixed.

In [ ]:
PROMPT = "ROMEO:\\n"  # @param {type:"string"}
CELL = "gru"  # @param ["rnn", "lstm", "gru"]
LENGTH = 300  # @param {type:"integer"}
TEMPERATURE = 0.8  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

torch.manual_seed(SEED)
print(generate_text(char_models[CELL], vocab, PROMPT.replace("\\n", "\n"), LENGTH, TEMPERATURE))

The output looks like a play: names in capitals, line breaks, mostly real words and the odd
invented one. It does not make sense, and nobody should expect it to. What the model has learned
is the statistics of the characters, and a character model has to spend most of its capacity on
spelling before it gets to anything else.

---

## Part 7 — From characters to words

The same model can read words instead of characters. The training loop, the shift-by-one targets
and the cell are unchanged. Three things change:

1. **Tokenisation**, lab 3's subject. We lowercase the text and split it into words, single
   punctuation marks and newlines. *o'er* and *thou'rt* stay whole. Keeping the newline as a token means the model can
   still learn the layout of a play.
2. **A much larger vocabulary, and an unknown token.** 65 characters becomes about 4,300 word
   types. A word must appear at least three times in TRAIN to get its own id, and every rarer word
   becomes the single token `<unk>`. The vocabulary is built from TRAIN only, so the test text can
   contain words the model has never seen, as real text does.
3. **An embedding instead of a one-hot vector.** With 4,300 types a one-hot input is wasteful.
   `nn.Embedding` is a learned table $E$ with one row of 128 numbers per word, and reading a word
   means looking up its row. It starts random and is trained with everything else. This is the
   same object as lab 8's word vectors, learned here as a by-product of predicting the next word.

In [ ]:
UNK = "<unk>"
MIN_COUNT = 3   # a word needs this many appearances in TRAIN to get its own id


def tokenize(text: str) -> list[str]:
    """Lowercase, then split into words, single punctuation marks, and newlines."""
    return re.findall(r"[a-z]+(?:'[a-z]+)*|\n|[^\sa-z]", text.lower())


def split_tokens(tokens: list[str]):
    a, b = int(0.8 * len(tokens)), int(0.9 * len(tokens))
    return tokens[:a], tokens[a:b], tokens[b:]


class WordVocab:
    """Every word seen at least MIN_COUNT times in TRAIN gets an id; the rest are <unk> (id 0)."""

    def __init__(self, words):
        self.words = list(words)                                # id -> word
        self.stoi = {w: i for i, w in enumerate(self.words)}    # word -> id

    @classmethod
    def from_tokens(cls, tokens):
        counts = Counter(tokens)
        kept = sorted(w for w, n in counts.items() if n >= MIN_COUNT)
        return cls([UNK] + kept)

    def encode(self, tokens) -> torch.Tensor:
        return torch.tensor([self.stoi.get(w, 0) for w in tokens], dtype=torch.long)

    def __len__(self):
        return len(self.words)


def detokenize(tokens) -> str:
    """Glue tokens back into text: no space before punctuation or after a newline."""
    out = ""
    for w in tokens:
        if w == "\n" or re.fullmatch(r"[^\sa-z<]", w) or not out or out[-1] in "\n-":
            out += w
        else:
            out += " " + w
    return out


def show_word(w: str) -> str:
    return {"\n": "↵"}.get(w, w)


tokens = tokenize(text)
train_toks, val_toks, test_toks = split_tokens(tokens)
wvocab = WordVocab.from_tokens(train_toks)
wtrain, wval = wvocab.encode(train_toks), wvocab.encode(val_toks)

print(tokenize("ROMEO:\nIs she dead? O'er the hill"))
print(f"\nTRAIN {len(train_toks):,} · VAL {len(val_toks):,} · TEST {len(test_toks):,} tokens")
print(f"vocabulary: {len(wvocab):,} words seen at least {MIN_COUNT} times in TRAIN, "
      f"covering {(wtrain != 0).float().mean().item():.1%} of TRAIN")

### The model

`WordRNN` is `CharRNN` with the one-hot step replaced by the embedding. The cell multiplies each
128-number row by its own input matrix, so SLP3's $W x_t$ becomes $W E[x_t]$. Every word is now
squeezed through 128 numbers, which saves weights and pushes words that are used in similar
places towards similar rows.

The default cell is the LSTM. A word model benefits from remembering further back than a
character model — 35 words reach back about as far as 175 characters — and Part 4 showed which
cell keeps its gradient over that distance.

One more change matters. A word model has far more weights for the amount of text, so it starts
memorising TRAIN sooner: train loss keeps falling while val loss turns back up. `train_word`
therefore keeps a copy of the weights from the round with the lowest val loss and returns those.
This is the job the validation split exists for.

In [ ]:
class WordRNN(nn.Module):
    """e_t = E[word at step t] -> cell -> scores over the vocabulary."""

    def __init__(self, vocab_size: int, embed: int, hidden: int, cell: str = "lstm"):
        super().__init__()
        self.cell = cell
        self.embed = nn.Embedding(vocab_size, embed)
        # Start the rows small. PyTorch's default gives values of size ~1, which Adam, moving
        # a weight ~lr per round, barely changes in 2000 rounds.
        nn.init.normal_(self.embed.weight, std=0.1)
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = rnn_cls(input_size=embed, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, vocab_size)

    def forward(self, x, h=None):
        all_h, h = self.rnn(self.embed(x), h)
        return self.head(all_h), h


@torch.no_grad()
def generate_ids(model, ids: torch.Tensor, n: int, temperature: float = 0.8) -> list[int]:
    """Read the prompt, then sample a word, feed it back in, repeat. <unk> is never sampled."""
    was_training = model.training
    model.eval()
    logits, h = model(ids.unsqueeze(0).to(DEVICE))
    out = ids.tolist()
    for _ in range(n):
        scores = logits[0, -1] / temperature
        scores[0] = -float("inf")                      # never write <unk>
        nxt = torch.multinomial(torch.softmax(scores, -1), 1).item()
        out.append(nxt)
        logits, h = model(torch.tensor([[nxt]], device=DEVICE), h)
    model.train(was_training)
    return out


def train_word(cell="lstm", steps=2000, embed=128, hidden=256, seq_len=35, batch=64, lr=3e-3,
               clip=1.0, seed=0):
    torch.manual_seed(seed)
    x, y = get_batch(wtrain, 1, 12)
    print("one training example (the target is the input shifted by one word):")
    print("  input  x: " + " ".join(show_word(wvocab.words[i]) for i in x[0].tolist()))
    print("  target y: " + " ".join(show_word(wvocab.words[i]) for i in y[0].tolist()))

    model = WordRNN(len(wvocab), embed, hidden, cell).to(DEVICE)
    print(f"\n{CELLS[cell]}, embedding {embed}, hidden {hidden}, "
          f"{sum(p.numel() for p in model.parameters()):,} weights")
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    print(f"random guessing among {len(wvocab):,} words gives loss ln({len(wvocab)}) = {math.log(len(wvocab)):.2f}")
    print(f"\n{'round':>6} {'train loss':>11} {'val loss':>9} {'e^val':>7}")

    t0, best, best_step, best_state = time.time(), float("inf"), 0, None
    for step in range(1, steps + 1):
        x, y = get_batch(wtrain, batch, seq_len)
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, _ = model(x)
        loss = loss_fn(logits.reshape(-1, len(wvocab)), y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()

        if step % 200 == 0 or step == 1:
            with torch.no_grad():
                vx, vy = get_batch(wval, 512, seq_len)
                vx, vy = vx.to(DEVICE), vy.to(DEVICE)
                vloss = loss_fn(model(vx)[0].reshape(-1, len(wvocab)), vy.reshape(-1)).item()
            mark = ""
            if vloss < best:
                best, best_step, mark = vloss, step, "  ← best so far, kept"
                best_state = copy.deepcopy(model.state_dict())
            print(f"{step:>6} {loss.item():>11.3f} {vloss:>9.3f} {math.exp(vloss):>7.0f}{mark}")
        if step % 1000 == 0 or step == steps:
            prompt = wvocab.encode(["romeo", ":", "\n"])
            print(f"  ── what it writes after {step} rounds (prompt 'romeo:') ──")
            out = detokenize(wvocab.words[i] for i in generate_ids(model, prompt, 40))
            print("  " + out.replace("\n", "\n  "))
            print("  " + "─" * 48)

    print(f"\nTrained in {time.time() - t0:.0f}s. Kept the weights from round {best_step} (val loss {best:.3f}).")
    model.load_state_dict(best_state)
    model.eval()
    return model


word_model = train_word("lstm")

### Reading the training log

Random guessing among about 4,300 words gives a loss of 8.4. The `e^val` column is
$e^{\text{val loss}}$, the number of words the model is in effect choosing between, and it is
the **perplexity** of session 14, on the same scale as for an n-gram model. Watch where val loss
stops falling while train loss carries on. In the reference run val loss is lowest at round 800
(4.142, a perplexity of 63) and climbs to 4.377 by round 2000, while train loss falls from 3.84
to 3.09. From round 800 onwards the model is fitting TRAIN rather than learning anything that
carries over, and the weights `train_word` returns are the ones from round 800.

In [ ]:
class LastWordOnly:
    """A word bigram model, counted from TRAIN: lab 5's model, with <unk> for rare words."""

    def __init__(self, ids: list[int]):
        self.counts = defaultdict(Counter)
        for a, b in zip(ids, ids[1:]):
            self.counts[a][b] += 1

    def top_guesses(self, last_id: int, top: int):
        c = self.counts[last_id]
        total = sum(c.values()) or 1
        return [(i, n / total) for i, n in c.most_common(top)]

    def best_guess(self, last_id: int) -> int:
        c = self.counts[last_id]
        return c.most_common(1)[0][0] if c else 0


word_baseline = LastWordOnly(wtrain.tolist())


@torch.no_grad()
def test_word(model, seed=0):
    data = wvocab.encode(test_toks)
    L = 100
    n = (len(data) - 1) // L
    x = data[: n * L].view(n, L)
    y = data[1 : n * L + 1].view(n, L)
    logits, _ = model(x.to(DEVICE))
    logits = logits.cpu()
    loss = F.cross_entropy(logits.reshape(-1, len(wvocab)), y.reshape(-1)).item()
    top5 = logits.topk(5, -1).indices
    pred = top5[..., 0]
    acc = (pred == y).float().mean().item()
    acc5 = (top5 == y.unsqueeze(-1)).any(-1).float().mean().item()

    base_pred = torch.tensor([[word_baseline.best_guess(i) for i in row] for row in x.tolist()])
    base_acc = (base_pred == y).float().mean().item()

    # Accuracy on real words only: punctuation and newlines are easy points.
    is_word = torch.tensor([bool(re.fullmatch(r"[a-z].*", w)) for w in wvocab.words])[y]
    word_acc = (pred == y)[is_word].float().mean().item()
    base_word_acc = (base_pred == y)[is_word].float().mean().item()

    print(f"graded on {n * L:,} test tokens\n")
    print(f"{'':32}{'all tokens':>11}{'words only':>12}")
    print(f"  {'random guessing (1 in ' + f'{len(wvocab):,})':<30}{1 / len(wvocab):>11.2%}{1 / len(wvocab):>12.2%}")
    print(f"  {'bigram, last word only':<30}{base_acc:>11.1%}{base_word_acc:>12.1%}   (no memory)")
    print(f"  {CELLS[model.cell]:<30}{acc:>11.1%}{word_acc:>12.1%}")
    print(f"\n  right answer in the model's top 5: {acc5:.1%}")
    print(f"  loss {loss:.2f}  ->  e^loss ≈ {math.exp(loss):.0f} words it is still choosing between "
          f"(out of {len(wvocab):,})")

    g = torch.Generator().manual_seed(seed)
    for _ in range(3):
        r = torch.randint(0, n, (1,), generator=g).item()
        s = torch.randint(0, L - 12, (1,), generator=g).item()
        actual = [show_word(wvocab.words[i]) for i in y[r, s : s + 12].tolist()]
        guess = [show_word(wvocab.words[i]) for i in pred[r, s : s + 12].tolist()]
        widths = [max(len(a), len(b)) for a, b in zip(actual, guess)]
        print("\n  actual : " + " ".join(a.ljust(w) for a, w in zip(actual, widths)))
        print("  guessed: " + " ".join(("·" if a == b else b).ljust(w) for a, b, w in zip(actual, guess, widths)))


test_word(word_model)

### Reading the result

The accuracy is far lower than the character model's, and that is expected rather than a
failure: each guess is now one of about 4,300 instead of one of 65. The fairer comparison is
again against the bigram model. In the reference run the LSTM gets 29.5% of all tokens right
against the bigram model's 23.5%, and 8.4% of real words against 5.3%. The "words only" column
leaves out punctuation and newlines, which are the easy points. The right word is among the
model's top five guesses 47.9% of the time.

**Try it.** Type a few words and see what the model expects next. `good morrow, my` is a good
first prompt. Words outside the vocabulary are read as `<unk>`, and the cell says which ones.

In [ ]:
WORDS = "good morrow, my"  # @param {type:"string"}


def encode_prompt(prompt: str) -> torch.Tensor:
    toks = tokenize(prompt.replace("\\n", "\n"))
    if not toks:
        raise ValueError("Give at least one word.")
    unknown = [w for w in toks if w not in wvocab.stoi]
    if unknown:
        print(f"(not in the vocabulary, read as {UNK}: {', '.join(unknown)})")
    return wvocab.encode(toks)


@torch.no_grad()
def top_word_guesses(model, ids: torch.Tensor, top: int):
    logits, _ = model(ids.unsqueeze(0).to(DEVICE))
    p, i = torch.softmax(logits[0, -1], -1).topk(top)
    return list(zip(i.tolist(), p.tolist()))


def fmt_word_guesses(guesses):
    return ", ".join(f"{show_word(wvocab.words[i])} {p:.0%}" for i, p in guesses)


ids = encode_prompt(WORDS)
print(f"Words so far: {' '.join(show_word(wvocab.words[i]) for i in ids.tolist())}\n")
print(f"  {CELLS[word_model.cell] + ':':<24}{fmt_word_guesses(top_word_guesses(word_model, ids, 5))}")
print(f"  bigram, last word only: {fmt_word_guesses(word_baseline.top_guesses(ids[-1].item(), 5))}")

And the word model can write, with the same temperature control as the character model.

In [ ]:
WORDS = "romeo :\\n"  # @param {type:"string"}
LENGTH = 80  # @param {type:"integer"}
TEMPERATURE = 0.8  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

torch.manual_seed(SEED)
print(detokenize(wvocab.words[i] for i in generate_ids(word_model, encode_prompt(WORDS), LENGTH, TEMPERATURE)))

Compare this with the character model's output. Every word is now a real word, because the model
can only choose from the vocabulary, and the lines have more of the sound of a sentence. They
still do not mean anything.

---

## Part 8 — What the embedding learned

Nobody told the model what any word means. Its embedding rows were trained only to make the next
word predictable. Lab 8 measured similarity between word vectors by cosine, and we can do the same
here: normalise every row of $E$ to length 1, and the cosine between two words is the dot product
of their rows. Words land close together when they are used in similar places, because then
swapping one for the other barely changes the model's next-word guesses. That is the
distributional hypothesis from session 21, arrived at by a different route.

In [ ]:
WORDS = "lord king she thou"  # @param {type:"string"}


@torch.no_grad()
def neighbours(model, words, top=8):
    E = F.normalize(model.embed.weight, dim=-1).cpu()   # every row scaled to length 1
    for word in words:
        word = word.lower()
        if word not in wvocab.stoi:
            print(f"{word}: not in the vocabulary\n")
            continue
        sims = E @ E[wvocab.stoi[word]]
        sims[wvocab.stoi[word]] = -1                    # skip the word itself
        s, ids = sims.topk(top)
        print(f"{word}: " + ", ".join(f"{show_word(wvocab.words[i])} {v:.2f}"
                                      for i, v in zip(ids.tolist(), s.tolist())))


neighbours(word_model, WORDS.split())

In the reference run the nearest words to *she* are *he* (0.81) and *it* (0.73), and the nearest
to *lord* are *sir*, *master* and *lady*: words that fill the same slot in a sentence. Further down
each list the neighbours are noisier than lab 8's, and they should be: lab 8's GloVe vectors were trained on
billions of tokens for exactly this purpose, and these were trained on a quarter of a million as a
side effect of something else. What the comparison shows is that word vectors are not a separate
invention. A network trained to predict the next word has to put words that behave alike in
similar places, and an embedding is where it puts them.

**Try it.** Try words of different kinds: a pronoun (*he*), a verb (*speak*), an archaic form
(*thee*), a name (*romeo*).

---

## What this lab established

1. A language model's training data is the text itself: the target is the input shifted by one.
2. Lab 5's bigram model, applied to characters, gets 26.5% of test characters right. The RNN gets
   47.1%, and the whole of that gap comes from the hidden state.
3. The vanishing gradient is measurable. Forty steps back, the plain RNN's gradient is about
   1e−5; the LSTM keeps 25 times more and the GRU over 100 times more.
4. `nn.RNN`, `nn.LSTM` and `nn.GRU` are interchangeable in one word. On this task the gated cells
   score about three points higher, and one run cannot separate the LSTM from the GRU.
5. Moving to words needs a vocabulary, an `<unk>` token and an embedding layer, and the embedding
   it learns places words that are used alike close together, as lab 8's vectors did.

To run any of this outside Colab, the scripts are in the course RNN repository,
<https://github.com/shripada/rnn>: `uv run demo/char_rnn.py train --cell lstm --details`
reproduces Part 5, and its README lists every command.

Lab 10 keeps the recurrent network and asks it to produce a whole sequence as output, rather than
one next token at a time from a prompt.